## ML-DSA Key Pair and Signature Generation

This notebook generates an ML-DSA key pair, uses the private key to digitally sign the ladder data, saves both the public key and the signature to files, and performs a verification test using the public key.

Specifically, this notebook obtains the authentication path (for a given leaf node) and the ladder **by parsing the prior ML-DSA full signature**. If a new key pair and a new signature are desired, the signature portion in mldsa_fullsig.txt **must be manually replaced** with the newly generated one.

### Importing Libraries

In [20]:
import struct
import socket
import os
import ctypes

### Configuration

In [21]:
# ML-DSA signature algorithm (should be supported by OpenSSL version 3.5.0):
sig_alg = "ML-DSA-44"

# Security parameter: this demo supports n = 16, 24, or 32.
n = 16

### Obtaining the Ladder to Create Signature

In [22]:
with open("data/mldsa_fullsig.txt", "r") as f:
    readin_val = f.read()

# The signature stored in full_sig.txt is supposed to be in hex format
fullsig = bytes.fromhex(readin_val)

#### The Authentication Path

In [23]:
from helper.mtl_parser import AuthPath

authpath = AuthPath(fullsig=fullsig, n=n)
authpath.authpath_print()

──────────────────────────────────────────────────
Authentication Path:
------------------------------
  Authentication Flags: 0000    - 2 bytes
------------------------------
  Series Identifier SID: 8F5CE818 A1DF091D FB8288CC AD6FF450 99A48C46 4F540CAB 05114981 736BB9D0    - 32 bytes
------------------------------
  Randomizer: C328BBA5 0C17738B 1F081ECE 4C0DB4C7    - 16 bytes
------------------------------
  Leaf Index: 00000000 00000004    - 8 bytes
------------------------------
  Target Rung Left Index: 00000000 00000000    - 8 bytes
------------------------------
  Target Rung Right Index: 00000000 00000007    - 8 bytes
------------------------------
  Sibling Hash Count: 3
------------------------------
  Sibling Node Hash Values:
    B1A6E1A4 DB73CE8D 9AC4449A B8F337B3    - 16 bytes
    5C91BF49 AF541835 32CD0584 162D7F8A    - 16 bytes
    D8D1F801 04C99E84 38ACF89A 8872237C    - 16 bytes
──────────────────────────────────────────────────
Level 0 Sibling:
  Left Index: 5
  Rig

#### The Ladder

In [24]:
from helper.mtl_parser import Ladder

ladder = Ladder(fullsig=fullsig, n=n, authpath_instance=authpath)
ladder.ladder_print()

──────────────────────────────────────────────────
Ladder:
------------------------------
  Ladder Flags: 0000    - 2 bytes
------------------------------
  Ladder Series Identifier SID: 8F5CE818 A1DF091D FB8288CC AD6FF450 99A48C46 4F540CAB 05114981 736BB9D0    - 32 bytes
------------------------------
  Ladder Rung Count: 2
------------------------------
  Ladder Rung:
------------------------------
    Rung Left Index: 00000000 00000000
    Rung Right Index: 00000000 00000007
    Rung Hash Value: FC1116C6 9A14E5AB 88ED3680 BDAC01FC
------------------------------
    Rung Left Index: 00000000 00000008
    Rung Right Index: 00000000 00000009
    Rung Hash Value: 4D846CCC C4586050 9E1A8189 0F3955CA
──────────────────────────────────────────────────


### Generating ML-DSA Key Pair

We use Python ctypes function interface to map native libcrypto library functions into Python handlers, enabling ML-DSA key generation and signature creation.

In [25]:
libcrypto = ctypes.CDLL("/usr/local/lib64/libcrypto.so.3")

In [26]:
# Define pointer and handlers

EVP_PKEY_p = ctypes.c_void_p
EVP_PKEY_CTX_p = ctypes.c_void_p
EVP_SIGNATURE_p = ctypes.c_void_p

# EVP_PKEY_CTX_new_from_name
libcrypto.EVP_PKEY_CTX_new_from_name.argtypes = [ctypes.c_void_p, ctypes.c_char_p, ctypes.c_char_p]
libcrypto.EVP_PKEY_CTX_new_from_name.restype = EVP_PKEY_CTX_p

# EVP_PKEY_keygen_init
libcrypto.EVP_PKEY_keygen_init.argtypes = [EVP_PKEY_CTX_p]
libcrypto.EVP_PKEY_keygen_init.restype = ctypes.c_int

# EVP_PKEY_keygen
libcrypto.EVP_PKEY_keygen.argtypes = [EVP_PKEY_CTX_p, ctypes.POINTER(EVP_PKEY_p)]
libcrypto.EVP_PKEY_keygen.restype = ctypes.c_int

# EVP_PKEY_CTX_new_from_pkey
libcrypto.EVP_PKEY_CTX_new_from_pkey.argtypes = [ctypes.c_void_p, EVP_PKEY_p, ctypes.c_char_p]
libcrypto.EVP_PKEY_CTX_new_from_pkey.restype = EVP_PKEY_CTX_p

# EVP_SIGNATURE_fetch
libcrypto.EVP_SIGNATURE_fetch.argtypes = [ctypes.c_void_p, ctypes.c_char_p, ctypes.c_char_p]
libcrypto.EVP_SIGNATURE_fetch.restype = EVP_SIGNATURE_p

# EVP_PKEY_sign_message_init
libcrypto.EVP_PKEY_sign_message_init.argtypes = [EVP_PKEY_CTX_p, EVP_SIGNATURE_p, ctypes.c_void_p]
libcrypto.EVP_PKEY_sign_message_init.restype = ctypes.c_int

# EVP_PKEY_verify_message_init
libcrypto.EVP_PKEY_verify_message_init.argtypes = [EVP_PKEY_CTX_p, EVP_SIGNATURE_p, ctypes.c_void_p]
libcrypto.EVP_PKEY_verify_message_init.restype = ctypes.c_int

# EVP_PKEY_sign
libcrypto.EVP_PKEY_sign.argtypes = [
    EVP_PKEY_CTX_p, 
    ctypes.c_void_p, ctypes.POINTER(ctypes.c_size_t), 
    ctypes.c_char_p, ctypes.c_size_t
]
libcrypto.EVP_PKEY_sign.restype = ctypes.c_int

# EVP_PKEY_verify
libcrypto.EVP_PKEY_verify.argtypes = [
    EVP_PKEY_CTX_p, 
    ctypes.c_char_p, ctypes.c_size_t, 
    ctypes.c_char_p, ctypes.c_size_t
]
libcrypto.EVP_PKEY_verify.restype = ctypes.c_int

libcrypto.EVP_PKEY_get_raw_public_key.argtypes = [
    EVP_PKEY_p,                  # pkey
    ctypes.c_char_p,             # buffer
    ctypes.POINTER(ctypes.c_size_t) # sizeof pointer
]
libcrypto.EVP_PKEY_get_raw_public_key.restype = ctypes.c_int

libcrypto.i2d_PUBKEY.argtypes = [ctypes.c_void_p, ctypes.POINTER(ctypes.c_void_p)]
libcrypto.i2d_PUBKEY.restype = ctypes.c_int

libcrypto.d2i_PUBKEY.argtypes = [ctypes.POINTER(EVP_PKEY_p), ctypes.POINTER(ctypes.c_void_p), ctypes.c_long]
libcrypto.d2i_PUBKEY.restype = EVP_PKEY_p 


# Cleanup
libcrypto.EVP_PKEY_free.argtypes = [EVP_PKEY_p]
libcrypto.EVP_PKEY_CTX_free.argtypes = [EVP_PKEY_CTX_p]
libcrypto.EVP_SIGNATURE_free.argtypes = [EVP_SIGNATURE_p]

#### Generate ML-DSA Key Pair

We generate an ML-DSA key pair using the algorithm specified at the beginning of this notebook. The public portion of the key pair is then extracted and saved to data/new_mldsa_pubkey.txt, which can be read by the MTL Mode demo notebook. 

In [27]:
def generate_ml_dsa_keypair(sig_alg) -> EVP_PKEY_p:
    """
    Generate an ML-DSA key pair context.
    """
    print(f"Generating key pair for {sig_alg}...")

    # Create keygen context
    ctx = libcrypto.EVP_PKEY_CTX_new_from_name(None, sig_alg.encode("utf-8"), None)
    if not ctx:
        raise RuntimeError(f"Could not create context for {sig_alg}.")
    
    try:
        if libcrypto.EVP_PKEY_keygen_init(ctx) <= 0:
            raise RuntimeError("Keygen initialization failed.")
        
        pkey = EVP_PKEY_p()
        if libcrypto.EVP_PKEY_keygen(ctx, ctypes.byref(pkey)) <= 0:
            raise RuntimeError("Key generation failed.")

        print("Successfully generated ML-DSA key pair.")
        return pkey
    finally:
        libcrypto.EVP_PKEY_CTX_free(ctx)

In [28]:
def export_public_key(pkey: ctypes.c_void_p) -> bytes:
    """
    Serialize the EVP_PKEY public key into standard X.509 SPKI DER bytes.
    """
    pubkey_len = ctypes.c_size_t(0)
    res = libcrypto.EVP_PKEY_get_raw_public_key(pkey, None, ctypes.byref(pubkey_len))
    
    # OpenSSL returns 1 on success
    if res <= 0 or pubkey_len.value == 0:
        raise RuntimeError("Failed to determine public key size.")
    
    # Allocatebuffer size
    buf = ctypes.create_string_buffer(pubkey_len.value)
    
    # Fetch key bytes into the buffer
    if libcrypto.EVP_PKEY_get_raw_public_key(pkey, buf, ctypes.byref(pubkey_len)) <= 0:
        raise RuntimeError("Failed to extract public key bytes.")
        
    return buf.raw[:pubkey_len.value]

In [29]:
key_pair = generate_ml_dsa_keypair(sig_alg)
pubkey = export_public_key(key_pair)

hex_pubkey = pubkey.hex()
with open("data/new_mldsa_pubkey.txt", "w") as file:
    file.write(hex_pubkey)
print("Finished!")

Generating key pair for ML-DSA-44...
Successfully generated ML-DSA key pair.
Finished!


### Sign Ladder to Get Signature

Next, we use the generated key pair to sign the ladder data, saving the resulting signature to data/new_mldsa_signature.txt. This updated signature should replace the one inside mldsa_fullsig.txt. Note that **a four-byte field indicating the signature length** must be appended to the ladder data before concatenating the actual signature. 

In [30]:
def sign_message(keypair: EVP_PKEY_p, sig_alg: str, message: bytes) -> bytes:
    """
    Sign a message.
    """
    # Context bound to the key
    sctx = libcrypto.EVP_PKEY_CTX_new_from_pkey(None, keypair, None)
    # Fetch the ML-DSA signature engine instance
    alg_encode = libcrypto.EVP_SIGNATURE_fetch(None, sig_alg.encode("utf-8"), None)
    
    if not sctx or not alg_encode:
        raise RuntimeError("Failed to set up signature structure.")
        
    try:
        # FIPS 204 requires message-level initialization
        if libcrypto.EVP_PKEY_sign_message_init(sctx, alg_encode, None) <= 0:
            raise RuntimeError("Signature message init failed.")
            
        # Get signature buffer size
        sig_len = ctypes.c_size_t(0)
        if libcrypto.EVP_PKEY_sign(sctx, None, ctypes.byref(sig_len), message, len(message)) <= 0:
            raise RuntimeError("Failed to determine signature buffer size.")
            
        # Get signature
        sig_buf = ctypes.create_string_buffer(sig_len.value)
        if libcrypto.EVP_PKEY_sign(sctx, sig_buf, ctypes.byref(sig_len), message, len(message)) <= 0:
            raise RuntimeError("Signing failed.")

        print("Successfully created ML-DSA signature")
        return sig_buf.raw[:sig_len.value]
    finally:
        libcrypto.EVP_SIGNATURE_free(sig_alg)
        libcrypto.EVP_PKEY_CTX_free(sctx)

In [31]:
signature = sign_message(key_pair, sig_alg, ladder.val)
sig_hex = signature.hex()
with open("data/new_mldsa_sigature.txt", "w") as file:
    file.write(sig_hex)
print("Finished!")

Successfully created ML-DSA signature
Finished!


The following line checks the length of the underlying signature. This value must be converted into an 8-character hex string (representing a 4-byte value, such as '00000974') and **appended to the ladder data prior to concatenating the signature**.

In [32]:
len(signature)

2420

### Verification Test: Using the Generated Key Pair

Next, we test if the signature can be verified using the key pair generated above. 

In [33]:
def test_verify_signature(key_pair: EVP_PKEY_p, sig_alg: str, message: bytes, signature: bytes) -> bool:
    """
    Verify an ML-DSA signature against the given message.
    """
    vctx = libcrypto.EVP_PKEY_CTX_new_from_pkey(None, key_pair, None)
    alg_encode = libcrypto.EVP_SIGNATURE_fetch(None, sig_alg.encode("utf-8"), None)
    
    if not vctx or not alg_encode:
        return False
        
    try:
        if libcrypto.EVP_PKEY_verify_message_init(vctx, alg_encode, None) <= 0:
            return False
            
        # EVP_PKEY_verify returns 1 for valid, 0 for invalid, or negative values on error
        res = libcrypto.EVP_PKEY_verify(vctx, signature, len(signature), message, len(message))
        return res == 1
    finally:
        libcrypto.EVP_SIGNATURE_free(alg_encode)
        libcrypto.EVP_PKEY_CTX_free(vctx)

In [34]:
# Verify
is_valid = test_verify_signature(key_pair, sig_alg, ladder.val, signature)
print(f"Is signature structurally valid? -> {is_valid}")

Is signature structurally valid? -> True


### Verification Test: Rebuild the Public Key Struct

Next, we test if the signature can be verified using a public key reconstructed from the raw public key bytes stored earlier.

In [35]:
with open("data/new_mldsa_pubkey.txt", "r") as f:
    pubkey_hex = f.read().strip()
pubkey_bytes = bytes.fromhex(pubkey_hex)

In [36]:
def reconstruct_pubkey(pubkey: bytes, sig_alg: str) -> EVP_PKEY_p:
    """
    Construct a public key back into an OpenSSL public-only EVP_PKEY.
    """        
    ctx = libcrypto.EVP_PKEY_CTX_new_from_name(None, sig_alg.encode("utf-8"), None)
    if not ctx:
        raise RuntimeError(f"Could not build context for {sig_alg}.")
    
    tmp_pkey = EVP_PKEY_p()
    pubkey_len = len(pubkey)
    try:
        if libcrypto.EVP_PKEY_keygen_init(ctx) <= 0 or libcrypto.EVP_PKEY_keygen(ctx, ctypes.byref(tmp_pkey)) <= 0:
            raise RuntimeError(f"Failed to compile metadata for {sig_alg}.")
        
        # Get structure length
        expected_der_len = libcrypto.i2d_PUBKEY(tmp_pkey, None)
        
        # Allocate buffer 
        buf = ctypes.create_string_buffer(expected_der_len)
        buf_ptr = ctypes.cast(buf, ctypes.c_void_p)
        buf_ref = ctypes.byref(buf_ptr)
        libcrypto.i2d_PUBKEY(tmp_pkey, buf_ref)
        
        # Extract ASN.1 header block
        # X.509 SPKI container has a metadata footprint of size (total DER size - key payload)
        header_size = expected_der_len - pubkey_len
        dynamic_header = buf.raw[:header_size]
        
    finally:
        # Free pointers
        if tmp_pkey:
            libcrypto.EVP_PKEY_free(tmp_pkey)
        libcrypto.EVP_PKEY_CTX_free(ctx)

    # Put together
    wrapped_pubkey = dynamic_header + pubkey

    # Feed the X.509 DER package into OpenSSL's d2i engine
    total_len = len(wrapped_pubkey)
    res_buf = ctypes.create_string_buffer(wrapped_pubkey)
    res_buf_ptr = ctypes.cast(res_buf, ctypes.c_void_p)
    res_buf_ref = ctypes.byref(res_buf_ptr)
    
    pubkey_construct = libcrypto.d2i_PUBKEY(None, res_buf_ref, total_len)
    if not pubkey_construct:
        raise RuntimeError(f"OpenSSL failed to parse spliced X.509 bytes for {sig_alg}.")
        
    return pubkey_construct

In [37]:
def verify_signature(pubkey: EVP_PKEY_p, sig_alg: str, message: bytes, signature: bytes) -> bool:
    """
    Verifies a signature using the loaded public key structure.
    """
    vctx = libcrypto.EVP_PKEY_CTX_new_from_pkey(None, pubkey, None)
    alg_encode = libcrypto.EVP_SIGNATURE_fetch(None, sig_alg.encode("utf-8"), None)

    if not vctx or not alg_encode:
        if vctx: libcrypto.EVP_PKEY_CTX_free(vctx)
        if alg_encode: libcrypto.EVP_SIGNATURE_free(alg_encode)
        return False

    try:
        if libcrypto.EVP_PKEY_verify_message_init(vctx, alg_encode, None) <= 0:
            return False
            
        res = libcrypto.EVP_PKEY_verify(vctx, signature, len(signature), message, len(message))
        return res == 1
        
    finally:
        libcrypto.EVP_SIGNATURE_free(alg_encode)
        libcrypto.EVP_PKEY_CTX_free(vctx)
        if pubkey:
            libcrypto.EVP_PKEY_free(pubkey)

In [38]:
# Verify
pubkey_rebuild = reconstruct_pubkey(pubkey_bytes, sig_alg)
#try:
is_valid = verify_signature(pubkey_rebuild, sig_alg, ladder.val, signature)
print(f"Is signature structurally valid? -> {is_valid}")

Is signature structurally valid? -> True
